<a href="https://colab.research.google.com/github/Claud1601/CN7030-Assignment/blob/main/_Forest_CoverType.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning on Big Data (CN7030)

## Group ID: Group 22

1. **Student 1:*Silakshana Dinesh*   
2. **Student 2:*Hemanth Sai Gude*   
3. **Student 3:*Vishal Bhai*
4. **Student 4:*Priya Sri Batchu*

**Project Title:** Multiclass Forest Cover Type Prediction Using Ensemble Machine Learning with PySpark



## Initiate and Configure Spark


In [1]:
 # ============================================================
# CN7030 - Machine Learning on Big Data
# Forest CoverType Multiclass Classification
# ============================================================

!pip install pyspark -q

from pyspark.sql import SparkSession


if 'spark' in locals() and spark.sparkContext._jsc is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("CN7030_Forest_CoverType")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark Version:", spark.version)
print("Application Name:", spark.sparkContext.appName)
print("Default Parallelism:", spark.sparkContext.defaultParallelism)

Spark Version: 4.0.4
Application Name: CN7030_Forest_CoverType
Default Parallelism: 2


# Task 1 Data chosen and Research objectives (10 Marks)

### 1.1 Selected Dataset

The dataset selected for this project is the **Forest CoverType dataset**. The objective is to predict the dominant forest cover type of a geographical area using cartographic and environmental attributes.

The dataset contains approximately **581,012 observations**, 54 predictor variables and one multiclass target variable named *Cover_Type*.

The target contains seven forest-cover classes:

1. Spruce/Fir
2. Lodgepole Pine
3. Ponderosa Pine
4. Cottonwood/Willow
5. Aspen
6. Douglas-fir
7. Krummholz

The predictor variables contain elevation, slope, aspect, hydrological distances, road distance, hillshade measurements, fire-point distance, wilderness-area indicators and soil-type indicators.

Although the dataset is smaller than some modern terabyte-scale big-data datasets, it contains more than half a million observations and 54 predictor variables. The complete machine-learning workflow is therefore implemented using PySpark DataFrames and Spark ML so that the same pipeline can scale to larger distributed datasets without redesigning the analysis.

### 1.2 Dataset Variables

The dataset contains ten quantitative cartographic attributes:

- *Elevation* - elevation of the geographical cell in metres.
- *Aspect* - direction of slope in degrees.
- *Slope* - slope steepness in degrees.
- *Horizontal_Distance_To_Hydrology* - horizontal distance to the nearest surface-water feature.
- *Vertical_Distance_To_Hydrology* - vertical distance to the nearest surface-water feature.
- *Horizontal_Distance_To_Roadways* - horizontal distance to the nearest roadway.
- *Hillshade_9am* - hillshade index measured at 9:00 AM.
- *Hillshade_Noon* - hillshade index measured at noon.
- *Hillshade_3pm* - hillshade index measured at 3:00 PM.
- *Horizontal_Distance_To_Fire_Points* - horizontal distance to the nearest wildfire ignition point.

Four binary wilderness-area variables are included:

- *Wilderness_Area1*
- *Wilderness_Area2*
- *Wilderness_Area3*
- *Wilderness_Area4*

A value of 1 indicates that the observation belongs to the corresponding wilderness area, whereas 0 indicates that it does not.

Forty binary soil-type variables are provided: *Soil_Type1* through *Soil_Type40*.

Each soil variable is one-hot encoded. A value of 1 indicates the presence of that soil type for the geographical observation and 0 indicates its absence.

The dependent variable is:

- *Cover_Type* - multiclass target variable containing values from 1 to 7.

### 1.3 Research Questions

**RQ1:** How accurately can forest cover type be predicted from cartographic and environmental variables using PySpark machine-learning algorithms?

**RQ2:** Does a Random Forest ensemble classifier provide better multiclass prediction performance than a single Decision Tree classifier?

**RQ3:** To what extent does hyperparameter optimisation improve the performance of the Random Forest model?

**RQ4:** How does class imbalance affect model performance, particularly for minority forest-cover classes?

**RQ5:** Which environmental and geographical variables contribute most to prediction of forest cover type?

**RQ6:** How stable and reproducible is the final machine-learning model across multiple random train-test splits?

### 1.4 Research Objectives

1. To load and process the Forest CoverType dataset using PySpark DataFrames.
2. To investigate missing values, duplicate observations and class imbalance.
3. To develop a multiclass Decision Tree classifier as a baseline model.
4. To develop a Random Forest classifier to demonstrate ensemble learning.
5. To optimise model hyperparameters using Spark ML model-selection techniques.
6. To evaluate the models using accuracy, F1-score, precision, recall and a multiclass confusion matrix.
7. To investigate model bias and variance.
8. To evaluate model reproducibility by reporting mean performance and standard deviation.
9. To identify the most influential predictor variables.
10. To discuss relevant Legal, Social, Ethical and Professional issues.

### 1.5 Proposed Big Data PySpark Pipeline

`text
Raw Forest CoverType Data
        ↓
Spark DataFrame
        ↓
Schema and Data Quality Analysis
        ↓
Missing-Value Analysis
        ↓
Duplicate Analysis
        ↓
Class Distribution Analysis
        ↓
Label Transformation
        ↓
VectorAssembler
        ↓
Train/Test Split
        ↓
Decision Tree Baseline
        ↓
Random Forest Ensemble
        ↓
Hyperparameter Optimisation
        ↓
Multiclass Prediction
        ↓
Accuracy / Precision / Recall / F1
        ↓
Confusion Matrix
        ↓
Repeated Experiments
        ↓
Mean ± Standard Deviation
        ↓
Feature Importance
        ↓
Results and LSEP Analysis
`


# Task 2 - Data Loading and Preprocessing & Visualisation (15 marks)


In [2]:
# Identify the students who made contributions and mention their names here.
# Contributors:
# Student 1: Silakshana Dinesh 3263086


import urllib.request
import os
import pandas as pd
import matplotlib.pyplot as plt

dataset_url = "https://kdd.ics.uci.edu/databases/covertype/covtype.data.gz"
dataset_path = "/content/covtype.data.gz"

if not os.path.exists(dataset_path):
    urllib.request.urlretrieve(dataset_url, dataset_path)

print("Dataset successfully downloaded.")
print("Dataset location:", dataset_path)


Dataset successfully downloaded.
Dataset location: /content/covtype.data.gz


In [3]:
# Define the dataset column names

quantitative_cols = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]

wilderness_cols = [
    "Wilderness_Area1",
    "Wilderness_Area2",
    "Wilderness_Area3",
    "Wilderness_Area4"
]

soil_cols = [f"Soil_Type{i}" for i in range(1, 41)]

feature_cols = quantitative_cols + wilderness_cols + soil_cols
all_columns = feature_cols + ["Cover_Type"]

print("Predictor variables:", len(feature_cols))
print("Total columns:", len(all_columns))


Predictor variables: 54
Total columns: 55


In [ ]:
# Load the raw dataset using a PySpark DataFrame

df = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "true")
    .csv(dataset_path)
)

df = df.toDF(*all_columns)

print("Dataset loaded successfully.")
df.show(5)


In [ ]:
# Dataset dimensions and schema

row_count = df.count()
column_count = len(df.columns)

print("Number of rows:", row_count)
print("Number of columns:", column_count)

df.printSchema()


In [ ]:
# Descriptive statistics for the quantitative predictors

df.select(quantitative_cols).describe().show()


In [ ]:
# Missing-value analysis

from pyspark.sql.functions import col, sum as spark_sum

missing_df = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

print("Missing Values")
missing_df.show(vertical=True)


### Missing-Value Interpretation

Inspect the output above before final submission. If all values are zero, the following interpretation is appropriate:

> A distributed missing-value analysis was performed across all variables using PySpark. No missing observations were identified. Therefore, imputation was not required and all observations were retained for subsequent modelling.


In [ ]:
# Duplicate analysis

original_rows = df.count()
unique_rows = df.dropDuplicates().count()
duplicates = original_rows - unique_rows

print("Original rows:", original_rows)
print("Unique rows:", unique_rows)
print("Duplicate rows:", duplicates)


In [ ]:
# Class distribution and imbalance analysis

from pyspark.sql.functions import round as spark_round

class_distribution = (
    df.groupBy("Cover_Type")
    .count()
    .orderBy("Cover_Type")
)

total = df.count()

class_distribution = class_distribution.withColumn(
    "Percentage",
    spark_round((col("count") / total) * 100, 2)
)

class_distribution.show()


In [ ]:
# Visualise class distribution

class_pd = class_distribution.toPandas()

plt.figure(figsize=(9, 5))
plt.bar(
    class_pd["Cover_Type"].astype(str),
    class_pd["count"]
)
plt.xlabel("Forest Cover Type")
plt.ylabel("Number of Observations")
plt.title("Distribution of Forest Cover Classes")
plt.tight_layout()
plt.show()


### Class-Imbalance Analysis

The class-distribution analysis demonstrates that the target variable is imbalanced. Some forest-cover categories occur substantially more frequently than others. Therefore, overall accuracy alone is insufficient for model evaluation. Weighted precision, weighted recall, F1-score and class-specific performance are also considered.

The proposed mitigation strategy is to evaluate minority-class behaviour explicitly using per-class precision and recall and to compare the standard Random Forest with a class-weighted Random Forest. This avoids relying only on overall accuracy.


In [ ]:
# Convert labels from 1-7 to 0-6 for Spark ML

df = df.withColumn(
    "label",
    (col("Cover_Type") - 1).cast("double")
)

df.select("Cover_Type", "label").show(10)


In [ ]:
# Assemble all 54 predictors into a single features vector

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

model_df = assembler.transform(df)

# Keep target, label and feature vector
model_df = model_df.select(
    "features",
    "label",
    "Cover_Type"
)

model_df.cache()

print("Cached observations:", model_df.count())
model_df.show(5, truncate=False)


### Feature Scaling Decision

Feature standardisation is not applied to the main Decision Tree and Random Forest models. Tree-based algorithms split features using threshold rules and therefore do not require the predictors to be on the same numerical scale.


In [ ]:
# Reproducible 80/20 train-test split

train_df, test_df = model_df.randomSplit(
    [0.80, 0.20],
    seed=42
)

train_df.cache()
test_df.cache()

print("Training samples:", train_df.count())
print("Testing samples:", test_df.count())


In [ ]:
# Create inverse-frequency class weights for an additional imbalance-aware model

train_total = train_df.count()
num_classes = 7

class_counts_train = train_df.groupBy("label").count()

weights_df = class_counts_train.withColumn(
    "classWeight",
    train_total / (num_classes * col("count"))
).select("label", "classWeight")

weighted_train_df = train_df.join(
    weights_df,
    on="label",
    how="left"
)

print("Class weights:")
weights_df.orderBy("label").show()

weighted_train_df.select("label", "classWeight").show(10)


# Task 3 - Model Selection and Implementation (15 marks)


### Student 1: [Name - Student ID]

#### Decision Tree Baseline

A Decision Tree classifier is used as the baseline model. It can capture nonlinear relationships and provides a useful comparison against the Random Forest ensemble. A single tree can, however, have relatively high variance and may overfit the training data.


In [ ]:
# Contributors:
# Student 1: Silakshana Dinesh 3263086
# Decision Tree Classifier

from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=10,
    seed=42
)

dt_model = dt.fit(train_df)
dt_predictions = dt_model.transform(test_df)

dt_predictions.select(
    "label",
    "prediction",
    "probability"
).show(10)


### Student 2: Hemanth Sai Gude

#### Random Forest Ensemble Model

Random Forest is selected as the main ensemble-learning algorithm. Instead of relying on one Decision Tree, it combines predictions from multiple trees. This ensemble mechanism can improve generalisation and reduce the variance associated with an individual tree.


In [ ]:
# Student 2: Hemanth Sai Gude
# Random Forest Classifier

from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=80,
    maxDepth=15,
    maxBins=64,
    seed=42
)

rf_model = rf.fit(train_df)
rf_predictions = rf_model.transform(test_df)

rf_predictions.select(
    "label",
    "prediction",
    "probability"
).show(10)


In [ ]:
# Additional imbalance-aware Random Forest using class weights

weighted_rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    weightCol="classWeight",
    numTrees=80,
    maxDepth=15,
    maxBins=64,
    seed=42
)

weighted_rf_model = weighted_rf.fit(weighted_train_df)

weighted_rf_predictions = weighted_rf_model.transform(test_df)

weighted_rf_predictions.select(
    "label",
    "prediction",
    "probability"
).show(10)


# Task 4 - Model Parameter Tuning (20 marks)


### Student 2:  — Decision Tree Hyperparameter Tuning


In [ ]:
# Student 2:
# Decision Tree Hyperparameter Tuning

from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

dt_tuning = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    seed=42
)

dt_param_grid = (
    ParamGridBuilder()
    .addGrid(dt_tuning.maxDepth, [5, 10, 15])
    .addGrid(dt_tuning.maxBins, [32, 64])
    .build()
)

print("Decision Tree parameter combinations:", len(dt_param_grid))

dt_tvs = TrainValidationSplit(
    estimator=dt_tuning,
    estimatorParamMaps=dt_param_grid,
    evaluator=f1_evaluator,
    trainRatio=0.8,
    parallelism=2,
    seed=42
)

dt_tvs_model = dt_tvs.fit(train_df)
best_dt_model = dt_tvs_model.bestModel

print("Best Decision Tree parameters:")
print(best_dt_model.extractParamMap())


### Student 2: [Name - Student ID] — Random Forest Hyperparameter Tuning


In [ ]:
# Student 2: [Name - Student ID]
# Random Forest Hyperparameter Tuning

rf_tuning = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    seed=42
)

rf_param_grid = (
    ParamGridBuilder()
    .addGrid(rf_tuning.numTrees, [50, 100])
    .addGrid(rf_tuning.maxDepth, [10, 15])
    .addGrid(rf_tuning.maxBins, [32, 64])
    .build()
)

print("Random Forest parameter combinations:", len(rf_param_grid))

rf_tvs = TrainValidationSplit(
    estimator=rf_tuning,
    estimatorParamMaps=rf_param_grid,
    evaluator=f1_evaluator,
    trainRatio=0.8,
    parallelism=2,
    seed=42
)

rf_tvs_model = rf_tvs.fit(train_df)
best_rf_model = rf_tvs_model.bestModel

print("Best Random Forest parameters:")
print(best_rf_model.extractParamMap())


### Hyperparameter-Tuning Rationale

The parameter grid is deliberately compact to keep the experiment computationally practical while still testing model-complexity parameters.

For the Decision Tree, *maxDepth* and *maxBins* are tuned.

For the Random Forest, *numTrees*, *maxDepth* and *maxBins* are tuned. F1-score is used as the model-selection metric because the target classes are imbalanced.

*TrainValidationSplit* is used as a computationally efficient first-stage tuning approach. A larger k-fold *CrossValidator* can be used if additional runtime is available.


# Task 5 - Model Evaluation and Accuracy Calculation (20 marks)


In [ ]:
# Common multiclass evaluation function

def evaluate_predictions(predictions):
    results = {}

    metrics = [
        "accuracy",
        "f1",
        "weightedPrecision",
        "weightedRecall"
    ]

    for metric in metrics:
        evaluator = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName=metric
        )
        results[metric] = evaluator.evaluate(predictions)

    return results


### Student 1: [Name - Student ID] — Tuned Decision Tree Evaluation


In [ ]:
# Student 1: Silakshana Dinesh 3263086

best_dt_predictions = dt_tvs_model.transform(test_df)
dt_results = evaluate_predictions(best_dt_predictions)

print("Tuned Decision Tree Results")
print("-" * 35)

for metric, value in dt_results.items():
    print(f"{metric}: {value:.4f}")


### Student 2:  — Tuned Random Forest Evaluation


In [ ]:
# Student 2: [Name - Student ID]

best_rf_predictions = rf_tvs_model.transform(test_df)
rf_results = evaluate_predictions(best_rf_predictions)

print("Tuned Random Forest Results")
print("-" * 35)

for metric, value in rf_results.items():
    print(f"{metric}: {value:.4f}")


In [ ]:
# Evaluate the imbalance-aware Random Forest for comparison

weighted_rf_results = evaluate_predictions(weighted_rf_predictions)

print("Weighted Random Forest Results")
print("-" * 35)

for metric, value in weighted_rf_results.items():
    print(f"{metric}: {value:.4f}")


### Evaluation Metrics

The following multiclass evaluation measures are reported:

- **Accuracy:** proportion of all test observations classified correctly.
- **Weighted Precision:** precision averaged across classes according to class frequency.
- **Weighted Recall:** recall averaged across classes according to class frequency.
- **F1-score:** harmonic mean of precision and recall.

Because the target classes are imbalanced, accuracy is interpreted together with F1-score, weighted metrics and class-specific results.


In [ ]:
# Confusion matrix counts for the tuned Random Forest

confusion_df = (
    best_rf_predictions
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

confusion_df.show(100)


In [ ]:
# Per-class precision and recall

per_class_results = []

for class_id in range(7):
    precision_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="precisionByLabel",
        metricLabel=float(class_id)
    )

    recall_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="recallByLabel",
        metricLabel=float(class_id)
    )

    precision = precision_evaluator.evaluate(best_rf_predictions)
    recall = recall_evaluator.evaluate(best_rf_predictions)

    per_class_results.append({
        "Cover Type": class_id + 1,
        "Precision": precision,
        "Recall": recall
    })

per_class_pd = pd.DataFrame(per_class_results)
per_class_pd


### Reproducibility Requirement

The final Random Forest configuration is run five times using fixed, pre-defined random seeds. Mean and standard deviation are reported so that the final result represents both average predictive performance and model stability rather than a single random split.


In [ ]:
# Extract selected Random Forest parameters robustly

best_num_trees = best_rf_model.getOrDefault(best_rf_model.numTrees)
best_max_depth = best_rf_model.getOrDefault(best_rf_model.maxDepth)
best_max_bins = best_rf_model.getOrDefault(best_rf_model.maxBins)

print("Selected numTrees:", best_num_trees)
print("Selected maxDepth:", best_max_depth)
print("Selected maxBins:", best_max_bins)


In [ ]:
# Five reproducible runs

import numpy as np

seeds = [42, 52, 62, 72, 82]
run_results = []

for seed in seeds:
    print(f"Running seed {seed}...")

    train_run, test_run = model_df.randomSplit(
        [0.80, 0.20],
        seed=seed
    )

    final_rf = RandomForestClassifier(
        labelCol="label",
        featuresCol="features",
        numTrees=best_num_trees,
        maxDepth=best_max_depth,
        maxBins=best_max_bins,
        seed=seed
    )

    final_model = final_rf.fit(train_run)
    final_predictions = final_model.transform(test_run)

    metrics = evaluate_predictions(final_predictions)

    run_results.append({
        "Seed": seed,
        "Accuracy": metrics["accuracy"],
        "F1": metrics["f1"],
        "Precision": metrics["weightedPrecision"],
        "Recall": metrics["weightedRecall"]
    })

runs_pd = pd.DataFrame(run_results)
runs_pd


In [ ]:
# Mean and standard deviation

summary = pd.DataFrame({
    "Metric": ["Accuracy", "F1", "Precision", "Recall"],
    "Mean": [
        runs_pd["Accuracy"].mean(),
        runs_pd["F1"].mean(),
        runs_pd["Precision"].mean(),
        runs_pd["Recall"].mean()
    ],
    "Standard Deviation": [
        runs_pd["Accuracy"].std(),
        runs_pd["F1"].std(),
        runs_pd["Precision"].std(),
        runs_pd["Recall"].std()
    ]
})

summary


In [ ]:
print("FINAL REPRODUCIBILITY RESULTS")
print("=" * 45)

for metric in ["Accuracy", "F1", "Precision", "Recall"]:
    mean_value = runs_pd[metric].mean()
    sd_value = runs_pd[metric].std()
    print(f"{metric}: {mean_value:.4f} ± {sd_value:.4f}")


# Task 6 - Results Visualization or Printing (5 marks)


### Student 1: [Name - Student ID] — Model Comparison


In [ ]:
# Student 1: [Name - Student ID]

model_comparison = pd.DataFrame({
    "Model": [
        "Tuned Decision Tree",
        "Tuned Random Forest",
        "Weighted Random Forest"
    ],
    "Accuracy": [
        dt_results["accuracy"],
        rf_results["accuracy"],
        weighted_rf_results["accuracy"]
    ],
    "F1": [
        dt_results["f1"],
        rf_results["f1"],
        weighted_rf_results["f1"]
    ],
    "Precision": [
        dt_results["weightedPrecision"],
        rf_results["weightedPrecision"],
        weighted_rf_results["weightedPrecision"]
    ],
    "Recall": [
        dt_results["weightedRecall"],
        rf_results["weightedRecall"],
        weighted_rf_results["weightedRecall"]
    ]
})

model_comparison


In [ ]:
# Visual comparison of model performance

ax = model_comparison.set_index("Model").plot(
    kind="bar",
    figsize=(10, 5)
)

plt.ylim(0, 1)
plt.ylabel("Evaluation Score")
plt.title("Comparison of Multiclass Classification Models")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Student 2:  — Confusion Matrix


In [ ]:
# Student 2: [Name - Student ID]

confusion_pd = confusion_df.toPandas()

matrix = (
    confusion_pd
    .pivot(index="label", columns="prediction", values="count")
    .fillna(0)
    .reindex(index=range(7), columns=range(7), fill_value=0)
)

matrix


In [ ]:
# Confusion matrix visualisation

matrix_values = matrix.values

fig, ax = plt.subplots(figsize=(8, 7))
img = ax.imshow(matrix_values)

ax.set_xlabel("Predicted Cover Type")
ax.set_ylabel("Actual Cover Type")
ax.set_title("Random Forest Confusion Matrix")

ax.set_xticks(range(7))
ax.set_yticks(range(7))
ax.set_xticklabels(range(1, 8))
ax.set_yticklabels(range(1, 8))

for i in range(matrix_values.shape[0]):
    for j in range(matrix_values.shape[1]):
        ax.text(
            j, i, int(matrix_values[i, j]),
            ha="center", va="center"
        )

plt.colorbar(img)
plt.tight_layout()
plt.show()


In [ ]:
# Feature importance from the tuned Random Forest

feature_importances = best_rf_model.featureImportances.toArray()

importance_pd = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": feature_importances
}).sort_values("Importance", ascending=False)

importance_pd.head(15)


In [ ]:
# Plot top 15 feature importances

top15 = importance_pd.head(15)

plt.figure(figsize=(10, 7))
plt.barh(
    top15["Feature"][::-1],
    top15["Importance"][::-1]
)
plt.xlabel("Feature Importance")
plt.title("Top 15 Random Forest Feature Importances")
plt.tight_layout()
plt.show()


### Feature-Importance Interpretation

Feature importance indicates which variables contributed most strongly to the Random Forest's predictive decisions. It should not be interpreted as proof that a variable causes a particular forest cover type.


In [ ]:
# Bias-variance analysis at different tree depths

bias_variance_results = []

for depth in [5, 10, 20]:
    temp_rf = RandomForestClassifier(
        labelCol="label",
        featuresCol="features",
        numTrees=50,
        maxDepth=depth,
        seed=42
    )

    temp_model = temp_rf.fit(train_df)

    train_predictions = temp_model.transform(train_df)
    test_predictions = temp_model.transform(test_df)

    accuracy_eval = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy"
    )

    train_accuracy = accuracy_eval.evaluate(train_predictions)
    test_accuracy = accuracy_eval.evaluate(test_predictions)

    bias_variance_results.append({
        "MaxDepth": depth,
        "Training Accuracy": train_accuracy,
        "Testing Accuracy": test_accuracy
    })

bias_variance_pd = pd.DataFrame(bias_variance_results)
bias_variance_pd


In [ ]:
# Bias-variance visualisation

plt.figure(figsize=(8, 5))

plt.plot(
    bias_variance_pd["MaxDepth"],
    bias_variance_pd["Training Accuracy"],
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    bias_variance_pd["MaxDepth"],
    bias_variance_pd["Testing Accuracy"],
    marker="o",
    label="Testing Accuracy"
)

plt.xlabel("Maximum Tree Depth")
plt.ylabel("Accuracy")
plt.title("Bias-Variance Analysis")
plt.legend()
plt.tight_layout()
plt.show()


### Bias and Variance Discussion

A shallow tree depth may result in **high bias (underfitting)** when both training and test performance are low. Increasing tree depth increases model complexity and may improve fit. However, if training accuracy becomes substantially higher than testing accuracy, this indicates increasing **variance (overfitting)**.

Random Forest reduces variance by aggregating predictions across multiple decision trees. The final model is therefore selected using validation performance rather than training accuracy alone.


# Task 7 - LSEP Considerations (10 marks)


## Student 1: Ethical Issue — Algorithmic Bias

The primary ethical issue investigated is algorithmic bias resulting from class imbalance. The Forest CoverType dataset does not contain equal numbers of observations for all seven forest classes. Models optimised only for overall accuracy may therefore prioritise majority classes while producing weaker predictions for minority forest categories.

To mitigate this issue, the analysis reports F1-score, precision, recall and class-specific results in addition to overall accuracy. The confusion matrix is also used to identify classes that are systematically misclassified. An additional class-weighted Random Forest is trained to examine whether minority-class treatment can be improved.

Model results should not be interpreted as infallible environmental decisions. Where such models are used in real environmental-management applications, predictions should support rather than replace appropriate professional and ecological expertise.


## Student 2: Professional Issue — Reproducibility and Transparency

A major professional requirement in machine learning is reproducibility. Results that change unpredictably between executions can reduce confidence in an analytics system and make comparison between models unreliable.

For this project, explicit random seeds are defined for train-test splitting, Decision Tree construction, Random Forest training and model optimisation. The final Random Forest configuration is additionally executed across five defined seeds. Mean performance and standard deviation are reported to demonstrate both predictive ability and model stability.

All preprocessing operations, selected attributes, hyperparameters and evaluation metrics are documented in the notebook. No model results should be entered manually; reported metrics must be generated directly from executable PySpark code. This improves transparency, auditability and professional accountability.


## Student 3: Legal Issue — Dataset Use and Attribution

The dataset should be used in accordance with the terms and attribution requirements of its original source. The report should clearly identify the Forest CoverType dataset and retain appropriate source acknowledgement. The group should avoid presenting the dataset as its own collected material and should not remove provenance information when sharing the analysis.

The analysis does not contain personal data, but responsible data-governance principles still apply. The source, purpose of use, processing steps and limitations should be documented so that the use of the dataset is transparent and traceable.


## Student 4: Social Issue — Responsible Environmental Interpretation


Machine-learning predictions of forest cover may influence how users interpret environmental conditions. Misclassification could lead to incorrect assumptions if predictions are treated as definitive observations rather than statistical estimates.

The project therefore reports model limitations and class-specific performance. In practical environmental applications, predictions should be combined with domain expertise, field observations and appropriate quality controls. This reduces the risk of over-reliance on automated classification and supports responsible use of machine-learning outputs.


## Overall Results and Discussion

**Complete this section only after all model cells have been executed.**

Report the actual generated values for:

- Tuned Decision Tree accuracy, F1, weighted precision and weighted recall.
- Tuned Random Forest accuracy, F1, weighted precision and weighted recall.
- Weighted Random Forest results.
- Five-run mean ± standard deviation.
- Best Random Forest hyperparameters.
- Minority-class behaviour from per-class precision/recall.
- Most important predictor variables.
- Evidence of underfitting or overfitting from the bias-variance experiment.

Do not enter invented performance values. Interpret only the output generated by the notebook.


## Conclusion

**Complete this paragraph after executing the notebook and reviewing the actual outputs.**

The conclusion should state whether the Random Forest improved upon the Decision Tree baseline, whether the target performance level was achieved, how stable the model was across the five runs, how class imbalance affected performance, and what the main limitations and future improvements are.


In [ ]:

!pip3 install nbconvert -q


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

notebook_filename = "GROUP_22_CRWK_CN7030.ipynb"
notebook_path = f"/content/drive/MyDrive/Colab Notebooks/Group_22_CRWK_CN7030_Forest_CoverType.ipynb"


!jupyter nbconvert --to html "{notebook_path}" --output-dir "/content/"

print(f"Notebook '{notebook_filename}' successfully converted to HTML and saved in /content/")

In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/Group_22_CRWK_CN7030_Forest_CoverType.ipynb"

In [ ]:
!jupyter nbconvert --to html GROUP_22_CRWK_CN7030.ipynb

In [ ]:
!ls -lah

In [ ]:
!jupyter nbconvert --to html "Group_22_CRWK_CN7030_Forest_CoverType.ipynb"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir())